<p style="font-size: 1.1rem; color:#9beb3b; text-align: center; font-family: monospace ">
    <strong>
       Importando bibliotecas
    </strong>
</p>

In [1]:
import pandas as pd
from openpyxl import Workbook, load_workbook
from openpyxl.styles import numbers #trabalhar com fontes
import xlwings as xw #abrir e fechar o arquivo para ativar as formulas do excel

<p style="font-size: 1.1rem; color:#9beb3b; text-align: center; font-family: monospace ">
    <strong><br/><br/>
       Criando planilha de controle de entregas
    </strong>
</p>

In [2]:
# local onde armazenará a planilha
control_path = 'dados/controle_entregas.xlsx'


wb = Workbook() #criar arquivo
ws = wb.active #criar planilha

ws.title = 'Entregas'

ws.append(
    [
        'Data',
        'Estado',
        'Motorista',
        'Entregas_Previstas',
        'Entregas_no_Prazo',
        'Entregas_Nao_Realizadas'
    ])

ws.append(['02/05/2025','SP','JOAO SOUZA','70','50','20'])
ws.append(['03/05/2025','RJ','MARIA SILVA','88','65','16'])
ws.append(['04/05/2025','RJ','JOSE ALVES','69','39','10'])
ws.append(['06/05/2025','DF','JOAO SOUZA','20','12','4'])
ws.append(['08/05/2025','AM','MARIA SILVA','10','5','2'])
ws.append(['10/05/2025','PB','MARIA SILVA','96','81','3'])
ws.append(['12/05/2025','BA','JOSE ALVES','41','32','10'])
ws.append(['12/05/2025','SP','JOAO SOUZA','36','26','8'])
ws.append(['12/05/2025','BA','JOSE ALVES','50','37','12'])

wb.save(control_path)
print('Planilha criada com sucesso!')

Planilha criada com sucesso!


<p style="font-size: 1.1rem; color:#9beb3b; text-align: center; font-family: monospace ">
    <strong><br/><br/>
       Inserindo novos indicadores
    </strong>
</p>

In [3]:
wb = load_workbook(control_path) #sempre utilizar para abrir o arquivo
ws = wb.active #para ativar as formulas no arquivo

ws['G1'] = 'Taxa de sucesso'
ws['H1'] = 'Taxa de atraso'
ws['I1'] = 'Entregas_Realizadas'

for linha in range(2, ws.max_row + 1):
        #coluna G   =     F2        / D2    (taxa de sucesso)
    ws[f'G{linha}'] = f'=F{linha}/D{linha}'
          # H2      =      1 -   G2 (Taxa de atraso)
    ws[f'H{linha}'] = f'=1-G{linha}'
    ws[f'I{linha}'] = f'=D{linha}-F{linha}'
 
wb.save(control_path)
print('Fomula inserida com sucesso!')

Fomula inserida com sucesso!


<p style="font-size: 1.1rem; color:#9beb3b; text-align: center; font-family: monospace ">
    <strong><br/><br/>
       Formatando as taxas como percentual
    </strong>
</p>

In [4]:
wb = load_workbook(control_path) #sempre utilizar para abrir o arquivo
ws = wb.active #para ativar as formulas no arquivo

for linha in range(2, ws.max_row + 1):
   ws[f'G{linha}'].number_format = '0.00%' 
   ws[f'H{linha}'].number_format = '0.00%' 

wb.save(control_path)
print('Calculo formato em percentual!')


Calculo formato em percentual!


<p style="font-size: 1.1rem; color:#9beb3b; text-align: center; font-family: monospace ">
    <strong><br/><br/>
        Resumo gerencial por estado
    </strong>
</p>

In [5]:
wb.save(control_path)

df = pd.read_excel(control_path, sheet_name='Entregas')

df['Taxa de sucesso 2'] = df['Entregas_Realizadas'] / df['Entregas_Previstas']

resumo_estado = df.groupby('Estado').agg({
    'Entregas_Previstas': 'sum', 
    'Entregas_Realizadas': 'sum',
    'Taxa de sucesso': 'mean',
    'Taxa de atraso': 'mean'
})

# resumo_estado
wb = load_workbook(control_path)

if 'Resumo' in wb.sheetnames:
    del wb['Resumo']

ws_resumo = wb.create_sheet('Resumo')

ws_resumo.append([
    'Estado',
    'Entregas_Previstas',
    'Entregas_Realizadas',
    'Taxa de sucesso',
    'Taxa de atraso'
])

for estado, valores in resumo_estado.iterrows():
    ws_resumo.append([
        estado,
        valores['Entregas_Previstas'],
        valores['Entregas_Realizadas'],
        valores['Taxa de sucesso'],
        valores['Taxa de atraso']
    ])
        
for linha in range(2,ws_resumo.max_row +1):
    ws_resumo[f'D{linha}'].number_format = '0.00%'
    ws_resumo[f'E{linha}'].number_format = '0.00%'

wb.save(control_path)




<p style="font-size: 1.1rem; color:#9beb3b; text-align: center; font-family: monospace ">
    <strong><br/><br/>
        Cálculos de taxas e inserindo no excel
    </strong>
</p>